# v8.6 — attack the decode reduction wall — the gate (CUDA-core, Colab T4)

v8 Cut 1 won (M-packing, 8.6×) but sits at ~10% HBM; Cut 2 (tensor cores) and v8.5 (double-buffer)
were both measured negatives. Triangulating them, the decode kernel is **compute-latency-bound** —
the wall is the per-key warp-shuffle reduction + the serial online-softmax recurrence, **not** the
KV-load latency. v8.6 tries to *hide* that latency with more in-flight work, as a **2-arm
single-variable ablation**, both CUDA-core / T4:

- **Arm 1 — occupancy** (`v8_gqa_occ`): stage KV as **FP16 smem** (16 KB, single buffer) → **4
  blocks/SM** (Cut 1 is 32 KB FP32 → 2). 2× resident warps to hide the latency (TLP).
- **Arm 2 — key-ILP** (`v8_gqa_ilp`): **KU=4-unrolled key loop** so the independent `__shfl_xor`
  reduction chains pipeline. FP32 smem kept → 2 blocks/SM unchanged (ILP is the only variable).

**Prediction (the deliverable):** the byte-roofline is **blind** (same `AI=2G/b`, same %HBM floor for
all three) — so this is a *schedule* claim the model can't see. Mechanistically: **Arm 1 (occupancy)
should be the stronger lever** (TLP hides the whole serial chain); **Arm 2 (ILP) weaker** (overlaps
only the reductions, leaves the serial softmax recurrence exposed). **If both are null → the floor is
the serial recurrence itself** → a score-stationary redesign is the real fix, and v9 FP8 stays
premature. Gate = both arms correct (+ Cut 1 not regressed) → the A/B reads which lever moved %HBM.

## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes --
#    importing torch first on a numpy-less venv (vast.ai) prints 'Failed to initialize NumPy'.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell (the old CPU torch stays loaded until restart).')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5
/usr/local/bin/python
shell python sees torch 2.11.0+cu128


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Roofline — UNCHANGED (`AI=2G/b`, same floor); the prediction is a SCHEDULE claim

Both arms keep identical bytes + FLOPs, so the model predicts the **same** HBM floor as Cut 1 — it is
blind to smem dtype and loop shape. The measured µs/tok and %HBM movement *is* the gap between this
byte-model and the real warp schedule.

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_75')   # the T4 we're running on
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s')
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'t_hbm floor':>12}")
for G in (1,2,4,8,16,32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.t_hbm*1e3:9.4f}ms')
print('\nRoofline is BLIND here: identical for v8_gqa / v8_gqa_occ / v8_gqa_ilp.')
print('Prediction: occupancy (Arm 1) > ILP (Arm 2); if both flat, the floor is the serial recurrence.')


arch: Tesla T4 | HBM 320.0 GB/s
  G |  AI=2G/b | limiter |  t_hbm floor
  1 |      1.0 |     HBM |    0.8390ms
  2 |      2.0 |     HBM |    0.4195ms
  4 |      4.0 |     HBM |    0.2098ms
  8 |      8.0 |     HBM |    0.1050ms
 16 |     16.0 |     HBM |    0.0525ms
 32 |     31.9 |     HBM |    0.0263ms

Roofline is BLIND here: identical for v8_gqa / v8_gqa_occ / v8_gqa_ilp.
Prediction: occupancy (Arm 1) > ILP (Arm 2); if both flat, the floor is the serial recurrence.


## 3. Build both arms (JIT) — `v8_gqa_occ` + `v8_gqa_ilp`

In [4]:
import glob, os, shutil
for name in ('fa_v8_gqa_occ', 'fa_v8_gqa_ilp'):
    for d in glob.glob(os.path.expanduser(f'~/.cache/torch_extensions/*/{name}')):
        if not glob.glob(os.path.join(d, '*.so')):
            shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
occ = build_kernel('v8_gqa_occ'); print('built Arm 1 (occupancy):', occ)
ilp = build_kernel('v8_gqa_ilp'); print('built Arm 2 (key-ILP):  ', ilp)


built Arm 1 (occupancy): <module 'fa_v8_gqa_occ' from '/root/.cache/torch_extensions/py312_cu128/fa_v8_gqa_occ/fa_v8_gqa_occ.so'>
built Arm 2 (key-ILP):   <module 'fa_v8_gqa_ilp' from '/root/.cache/torch_extensions/py312_cu128/fa_v8_gqa_ilp/fa_v8_gqa_ilp.so'>


## 4. Correctness gate — both arms + Cut 1 regression (Gate 1 of 2)

Both arms inherit the full GQA suite (decode G∈{1,2,4,8} × non-multiple `N_k` × d{64,128} × causal
both ways; idle-warp G=3 + multi-tile G=16; square reduction). `v8_gqa` re-run confirms no regression.

In [5]:
!python -m pytest tests/test_correctness.py -k "v8_gqa_occ or v8_gqa_ilp or v8_gqa" -q


........................................................................ [ 37%]
........................................................................ [ 75%]
..............................................                           [100%]
190 passed, 126 deselected in 307.44s (0:05:07)


## 5. THE A/B — both arms vs `v8_gqa` (Cut 1), same G-sweep

Reads the µs/tok + %HBM delta per arm, per G. A %HBM rise on Arm 1 confirms occupancy was the
hideable latency; flat on both = the first-class negative isolating the serial recurrence.

In [6]:
print('=== Cut 1: v8_gqa (single FP32 buffer, 2 blocks/SM) ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== Arm 1: v8_gqa_occ (FP16 smem, 4 blocks/SM) ===')
!python -m bench.harness --backend v8_gqa_occ --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== Arm 2: v8_gqa_ilp (KU=4 key-unroll, 2 blocks/SM) ===')
!python -m bench.harness --backend v8_gqa_ilp --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


=== Cut 1: v8_gqa (single FP32 buffer, 2 blocks/SM) ===
# device: Tesla T4 (sm_75)  clock~360/1590MHz  backend=v8_gqa  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v7_paged -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -fPIC -std=c++17 -c /content/flashattention-cuda/kernels/v7_paged/binding.cpp -o binding.o 
[2/3] /usr/local/cuda/bin/nvcc -MD -MF paged_attention.cuda.o.d -DTORCH_EXTENSION_NAME=fa_v7_paged -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -is

## 6. Reclaim-SDPA-at-batch (G=8) — does either arm widen the win over SDPA?

Cut 1 already beats SDPA at all batch; the question is whether hiding the reduction latency moves the
µs/tok floor at B≥8 (where the kernel is SM-saturated).

In [7]:
print('=== Cut 1: v8_gqa ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== Arm 1: v8_gqa_occ ===')
!python -m bench.harness --backend v8_gqa_occ --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== Arm 2: v8_gqa_ilp ===')
!python -m bench.harness --backend v8_gqa_ilp --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64


=== Cut 1: v8_gqa ===
# device: Tesla T4 (sm_75)  clock~480/1590MHz  backend=v8_gqa  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
ninja: no work to do.
     1x8x1x64/8192 G8 |   0.246/  0.305 |    30.73 |   2.7% |   11.50x |    2.43x | HBM (~0.01ms)
    1x8x1x128/8192 G8 |   0.240/  0.310 |    29.94 |   5.5% |    7.78x |    3.99x | HBM (~0.01ms)
     8x8x1x64/8192 G8 |   0.648/  0.719 |    10.12 |   8.1% |    8.40x |    7.63x | HBM (~0.05ms)
    8x8x1x128/8192 G8 |   1.070/  1.133 |    16.71 |   9.8% |    7.70x |    7.62x | HBM (~0.10ms)
    16x8x1x64/8192 G8 |   1.313/  1.379 |    10.25 |   8.0% |    8.28x |    7.50x | HBM (~0.10ms)
   16x8x1x128/8192 G8 |   2.195/  2.239 |    17.15 |   9.6% |    7.39x |    7.45x | HBM (~0.21ms)
    32x8x1x64/8192 G8 |   3.245/  3.322 |    12.67 |   6.5% |    6.45x |    6.04x | HBM (~0.21ms)
   32x8x1x128/8192 G8 |   5.212/  5.404 |    20